# VCuboidFIT · Tầng 0 + Tầng 1 trên Colab / Kaggle
Chạy thí nghiệm chọn 5% (Tầng 0 CPU, rồi Tầng 1 PointPillars GPU) trên một bộ nuScenes. Không dùng Docker; dùng `--sweeps 1` (chỉ cần keyframe). Hướng dẫn chi tiết: `docs/kaggle-colab.md`, thuê GPU: `docs/gpu-rental.md`.
Mọi bước **idempotent**: phiên bị ngắt thì mở lại, chạy lại từ ô 0 — phần đã có trong `EXP` được bỏ qua.

Chuẩn bị: (a) file dữ liệu nuScenes (`v1.0-mini.tgz` hoặc `vcf_part_*.zip`) chép vào `Drive/vcf_upload` — ô 0 tự giải nén; (b) mã nguồn lấy bằng `git clone` (`GIT_URL`/`GIT_REF` ở ô cấu hình) hoặc gói `vcf-cloud-<sha>.tar.gz` từ `bash scripts/cloud/pack.sh`. GPU: **T4 / L4 / A100, không dùng P100** (sm_60). Tầng 0 chạy được không cần GPU: bỏ qua phần kiểm GPU và Tầng 1.

## 0. Nạp dữ liệu (chỉ cần đưa file lên)
Chép file nuScenes (`v1.0-mini.tgz`, hoặc các `vcf_part_*.zip`) vào thư mục `Drive của tôi/vcf_upload`, hoặc để trống để upload trực tiếp. Ô dưới tự giải nén và đặt `NUSC_READY`; ô cấu hình phía sau dùng giá trị đó, không cần sửa đường dẫn.

In [ ]:
# Ô 0 - NẠP DỮ LIỆU: chỉ cần đưa file lên, ô này tự tìm, tự giải nén và nối vào cấu hình.
# Cách 1 (khuyên dùng, ổn định với file lớn): chép file vào thư mục UPLOAD_DIR trên Google Drive.
# Cách 2: để UPLOAD_DIR trống, ô sẽ hiện nút chọn file để upload trực tiếp (chậm với file vài GB).
# Chấp nhận: v1.0-mini.tgz / .tar / .zip của nuScenes, hoặc các part vcf_part_*.zip (vcf-pack).
# Đã có sẵn thư mục chứa v1.0-* thì bỏ qua. Chạy lại an toàn.
import glob, os, shutil, tarfile, zipfile
from google.colab import drive

drive.mount("/content/drive")
UPLOAD_DIR = "/content/drive/MyDrive/vcf_upload"   # thư mục Drive để thả file dữ liệu
DEST = "/content/nuscenes"                          # giải nén vào ổ tạm của Colab (nhanh)


def _safe(root, name):
    out = os.path.realpath(os.path.join(root, name))
    assert out == root or out.startswith(root + os.sep), f"đường dẫn không an toàn: {name}"
    return out


def _find_nusc(base, depth=3):
    """Thư mục (base hoặc con trong `depth` cấp) chứa v1.0-*, hoặc None."""
    for d in range(depth + 1):
        hit = glob.glob(os.path.join(base, *["*"] * d, "v1.0-*"))
        if hit:
            return os.path.dirname(sorted(hit)[0])
    return None


def _extract(path, root):
    n = 0
    if tarfile.is_tarfile(path):
        with tarfile.open(path) as tf:
            for m in tf:
                if not (m.isfile() or m.isdir()):
                    continue
                _safe(root, m.name)
                tf.extract(m, root)
                n += 1
    elif zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as zf:
            for m in zf.infolist():
                name = m.filename[5:] if m.filename.startswith("data/") else m.filename  # vcf-pack
                if m.is_dir() or not name:
                    continue
                out = _safe(root, name)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with zf.open(m) as s, open(out, "wb") as d:
                    shutil.copyfileobj(s, d)
                n += 1
    else:
        print("bỏ qua (không phải tar/zip):", path)
    return n


NUSC_READY = _find_nusc(DEST) or _find_nusc("/content/drive/MyDrive/nuscenes")
if NUSC_READY:
    print("Đã có dữ liệu ở", NUSC_READY, "- bỏ qua")
else:
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    files = sorted(f for f in glob.glob(UPLOAD_DIR + "/*") if os.path.isfile(f))
    if not files:
        from google.colab import files as colab_files
        print("Chưa thấy file trong", UPLOAD_DIR, "- hãy chọn file dữ liệu để upload:")
        up = colab_files.upload()
        files = [os.path.join("/content", k) for k in up]
    assert files, "Không có file dữ liệu nào"
    os.makedirs(DEST, exist_ok=True)
    root = os.path.realpath(DEST)
    total = sum(_extract(f, root) for f in files)
    NUSC_READY = _find_nusc(DEST)
    assert NUSC_READY, f"Giải nén {total} file nhưng không thấy thư mục v1.0-* trong {DEST}"
    print(f"Đã giải nén {total} file từ {len(files)} tệp")
print("NUSC_READY =", NUSC_READY, "|", sorted(os.listdir(NUSC_READY)))


## 1. Phát hiện môi trường và cấu hình đường dẫn
Sửa các biến ở đầu ô này cho khớp dữ liệu của bạn. `EXP` phải nằm trên chỗ lưu được (Kaggle: `/kaggle/working`; Colab: Drive).

In [ ]:
import os, sys, glob, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
ENV = "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local"
print("môi trường:", ENV)

# ---------- CẤU HÌNH (sửa ở đây) ----------
# Thư mục nuScenes chứa v1.0-mini/ hoặc v1.0-trainval/ (và samples/, sweeps/)
NUSC_KAGGLE = "/kaggle/input/nuscenes"                      # tên Kaggle Dataset của bạn
NUSC_COLAB  = globals().get("NUSC_READY", "/content/drive/MyDrive/nuscenes")  # ô 0 đặt sẵn
NUSC_LOCAL  = os.environ.get("NUSC", "/data/nuscenes")
# Gói mã nguồn: tar.gz do scripts/cloud/pack.sh tạo (tìm theo glob), hoặc git clone nếu có remote
PACK_GLOBS = ["/kaggle/input/*/vcf-cloud-*.tar.gz", "/content/drive/MyDrive/vcf-cloud-*.tar.gz",
              "/content/vcf-cloud-*.tar.gz", "./vcf-cloud-*.tar.gz"]
GIT_URL = "https://github.com/Nguyendev037/VCuboidFit.git"   # để trống = dùng tar.gz
GIT_REF = "fix/Linh"   # đổi thành "main" sau khi nhánh được merge
# -------------------------------------------

if ENV == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Không mount được Drive:", e)

WORK = Path("/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else Path.cwd())
NUSC = {"kaggle": NUSC_KAGGLE, "colab": NUSC_COLAB, "local": NUSC_LOCAL}[ENV]
# EXP phải sống sót khi phiên chết: Kaggle /kaggle/working (tải về từ Output), Colab -> Drive
EXP = str(Path("/content/drive/MyDrive/vcf_exp/mini") if ENV == "colab" else WORK / "exp")
Path(EXP).mkdir(parents=True, exist_ok=True)

REPO = WORK / "vcf"
if (REPO / "model" / "scripts" / "tier1.sh").exists():
    REPO = REPO / "model"
if not (REPO / "scripts" / "tier1.sh").exists():
    if GIT_URL:
        subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, GIT_URL, str(REPO)], check=True)
        REPO = REPO / "model"
    else:
        packs = sorted(p for g in PACK_GLOBS for p in glob.glob(g))
        assert packs, f"Không thấy vcf-cloud-*.tar.gz; thử {PACK_GLOBS} hoặc đặt GIT_URL"
        REPO.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "xzf", packs[-1], "-C", str(REPO)], check=True)
        print("giải nén", packs[-1])
REPO = str(REPO)
assert Path(NUSC).exists(), f"NUSC không tồn tại: {NUSC} (sửa NUSC_* ở trên)"
os.environ.update(NUSC=NUSC, EXP=EXP, REPO=REPO)
print("NUSC =", NUSC, "\nEXP  =", EXP, "\nREPO =", REPO, "\nnuScenes có:", sorted(os.listdir(NUSC))[:12])

## 2. Kiểm GPU
Cần CUDA 12.x có sẵn trong image Kaggle/Colab. **P100 (sm_60) không nằm trong danh sách arch** ⇒ đổi sang T4 (Kaggle: Accelerator → *GPU T4 x2*).

In [ ]:
import torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "không có nvidia-smi")
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available(), "Không có GPU: bật Accelerator GPU rồi chạy lại (Tầng 0 chạy CPU được: bỏ qua ô 3 và 5)."
CAP = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % CAP)
if CAP < (7, 0):
    raise SystemExit("GPU sm_%d%d (vd P100) KHÔNG được hỗ trợ — chọn T4/L4/A100." % CAP)
ARCH = "%d.%d" % CAP          # build CUDA ops đúng GPU đang chạy (T4=7.5, A100=8.0, L4=8.9)
os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
print("TORCH_CUDA_ARCH_LIST =", ARCH)

## 3. Cài phụ thuộc + OpenPCDet (commit ghim `233f849`)
Tải tarball qua HTTPS (không cần git), vá bằng `docker/tier1/patch_pcdet.py` (bỏ import Argo2, alias `np.int/float/bool`), build CUDA ops, ghi `BUILD_COMMIT` (train_seed đọc file này). Lần đầu ~10 phút. `numpy<2` có thể cần **khởi động lại runtime** nếu numpy đã được nạp (Colab sẽ báo), rồi chạy lại từ ô 1.

In [ ]:
%%bash
set -e
pip -q install spconv-cu120 "numpy<2" nuscenes-devkit easydict tensorboardX kornia==0.6.12 \
    scikit-learn joblib pyarrow pandas pyyaml opencv-python-headless pillow

In [ ]:
%%bash
set -e
COMMIT=233f849829b6ac19afb8af8837a0246890908755
cd "$(dirname "$REPO")"
if [ ! -f OpenPCDet/BUILD_COMMIT ]; then
  wget -qO pcdet.tgz https://github.com/open-mmlab/OpenPCDet/archive/$COMMIT.tar.gz
  rm -rf OpenPCDet "OpenPCDet-$COMMIT"
  tar xzf pcdet.tgz && mv "OpenPCDet-$COMMIT" OpenPCDet
  python "$REPO/docker/tier1/patch_pcdet.py" OpenPCDet
  (cd OpenPCDet && sed -i '/^av2/d;/^kornia/d;/^numpy/d;/^spconv/d' requirements.txt \
     && python setup.py develop)
  echo $COMMIT > OpenPCDet/BUILD_COMMIT
fi
cat OpenPCDet/BUILD_COMMIT

In [ ]:
PCDET_ROOT = str(Path(REPO).parent / "OpenPCDet")
os.environ["PCDET_ROOT"] = PCDET_ROOT
os.environ["PYTHONPATH"] = f"{REPO}/worker:{PCDET_ROOT}:" + os.environ.get("PYTHONPATH", "")
for p in (f"{REPO}/worker", PCDET_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)
import importlib; importlib.invalidate_caches()
import pcdet, pcdet.datasets   # nếu lỗi np.int / av2: patch_pcdet chưa chạy
print("pcdet", pcdet.__version__, "| PCDET_ROOT =", PCDET_ROOT)

## 4. Tầng 0 (CPU): descriptor hình học, chọn 5%, tune
Tạo `EXP/index.parquet` + `features/` — **bắt buộc** trước Tầng 1 (`tier1.sh` thoát mã 4 nếu thiếu).

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

## 5. Tầng 1 (GPU): PointPillars huấn luyện từ seed S + suy luận
`--sweeps 1` (chỉ keyframe — ghi rõ trong báo cáo, dùng cùng giá trị cho downstream). Hết VRAM: `--batch 2` rồi `1`. Checkpoint ở `EXP/t1/ckpt/seed_latest.pth`; chạy lại thì phần đã có được bỏ qua.

In [ ]:
%%bash
set -e
bash "$REPO/scripts/tier1.sh" "$NUSC" "$EXP" --no-docker --sweeps 1 --epochs 20 --batch 4

## 6. Chạy lại Tầng 0 để có ma trận Tầng 1
Lần này có `t1/signals.parquet` ⇒ ma trận thêm `hybrid_mmr`, `t1_*_mmr`, `entropy_only`.

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

In [ ]:
from IPython.display import Markdown, display
rep = Path(EXP) / "V" / "report_selection.md"
display(Markdown(rep.read_text(encoding="utf-8")) if rep.exists() else Markdown(f"Chưa có `{rep}`"))

## 7. Lưu kết quả
Zip `EXP` **không kèm** `t1/pcdet_data` (chỉ symlink + info tạm, tái tạo được). Kaggle: file ở `/kaggle/working` ⇒ *Save Version → Save & Run All (Commit)* hoặc tải ở tab Output. Colab: `EXP` đã ở Drive, thêm bản zip cạnh đó.

**Resume:** phiên ngắt ⇒ mở lại, chạy ô 1 → 3 (cài lại môi trường), giải nén zip cũ vào `EXP` nếu cần, rồi chạy tiếp ô 4–6; các bước đã xong tự bỏ qua.

In [ ]:
import zipfile, time
OUT_DIR = Path("/content/drive/MyDrive") if ENV == "colab" else WORK
zpath = OUT_DIR / f"vcf_exp_{time.strftime('%Y%m%d_%H%M')}.zip"
skip = Path(EXP) / "t1" / "pcdet_data"
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(EXP, followlinks=False):
        dirs[:] = [d for d in dirs if Path(root, d) != skip]
        for f in files:
            fp = Path(root, f)
            if fp.is_symlink():
                continue
            z.write(fp, fp.relative_to(Path(EXP).parent)); n += 1
print(f"{zpath}  {zpath.stat().st_size/1e6:.1f} MB, {n} file")

## 8. (Tuỳ chọn, mặc định tắt) Cổng G1
Đặt `RUN_G1 = True` để chạy `c4.cli.lidar_g1`. Chỉ bật khi đã đọc SPEC của cổng G1.

In [ ]:
RUN_G1 = False
if RUN_G1:
    r = subprocess.run([sys.executable, "-m", "c4.cli.lidar_g1", "--data-root", NUSC, "--out", EXP],
                       cwd=f"{REPO}/worker")
    print("exit", r.returncode)
else:
    print("G1 tắt (RUN_G1 = False)")